# Tomography and denoising
Implementation notebook. Supply your own SLphan.npy phantom; see README.md for dependencies.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import astra
except ImportError:
    import sys
    !{sys.executable} -m pip install astra-toolbox
    import astra
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio, structural_similarity
from scipy.fftpack import dct, idct
from scipy.linalg import svd
import pywt
from scipy.ndimage import laplace
import copy
import warnings
import sys
import io
warnings.filterwarnings("ignore")

# Tee: write all print() output to a log file AND the notebook simultaneously
class _Tee:
    def __init__(self, stream, filepath):
        self._screen = stream
        self._file = open(filepath, "w", encoding="utf-8")
    def write(self, data):
        self._screen.write(data)
        self._file.write(data)
    def flush(self):
        self._screen.flush()
        self._file.flush()
    def close(self):
        self._file.close()
    def __getattr__(self, name):
        return getattr(self._screen, name)

_tee = _Tee(sys.stdout, "notebook_output_log.txt")
sys.stdout = _tee
print("=== COMP0114 Notebook Output Log ===")
print(f"Captured at: {__import__("datetime").datetime.now().strftime("%Y-%m-%d %H:%M:%S")}")
print("=" * 40)


In [ ]:
from pathlib import Path
import re

EXPORT_DIR = Path('notebook_exported_images')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_COUNTER = {'value': 0}


def _next_fig_path(tag='figure'):
    FIG_COUNTER['value'] += 1
    safe = re.sub(r'[^a-zA-Z0-9]+', '_', str(tag)).strip('_').lower()
    if not safe:
        safe = 'figure'
    return EXPORT_DIR / f"figure_{FIG_COUNTER['value']:02d}_{safe}.png"


def show_img(img, title='', save_tag=None):
    plt.figure(figsize=(4, 3.5))
    plt.imshow(img, cmap='gray')
    plt.colorbar(label='Intensity')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    out_path = _next_fig_path(save_tag if save_tag is not None else title)
    plt.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.show()
    print(f'Saved figure: {out_path}')


def save_current_figure(tag):
    out_path = _next_fig_path(tag)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.show()
    print(f'Saved figure: {out_path}')


# Task 1: Astra Radon projection
def astra_radon(img, angles):
    n = img.shape[0]
    det_count = int(np.ceil(np.sqrt(2) * n))
    vol_geom = astra.create_vol_geom(n, n)
    proj_geom = astra.create_proj_geom('parallel', 1.0, det_count, angles)
    projector_id = astra.create_projector('strip', proj_geom, vol_geom)
    sinogram_id, sinogram = astra.create_sino(img, projector_id)
    astra.data2d.delete(sinogram_id)
    astra.projector.delete(projector_id)
    return sinogram.astype(np.float32)


# Task 1: Astra unfiltered backprojection (BP)
def astra_bp(sinogram, angles, output_shape):
    n = output_shape[0]
    det_count = sinogram.shape[1]
    vol_geom = astra.create_vol_geom(n, n)
    proj_geom = astra.create_proj_geom('parallel', 1.0, det_count, angles)
    projector_id = astra.create_projector('strip', proj_geom, vol_geom)
    rec_id = astra.data2d.create('-vol', vol_geom)
    sino_id = astra.data2d.create('-sino', proj_geom, sinogram.astype(np.float32))
    cfg = astra.astra_dict('BP')
    cfg['ReconstructionDataId'] = rec_id
    cfg['ProjectionDataId'] = sino_id
    cfg['ProjectorId'] = projector_id
    alg_id = astra.algorithm.create(cfg)
    astra.algorithm.run(alg_id)
    rec = astra.data2d.get(rec_id)
    astra.data2d.delete(rec_id)
    astra.data2d.delete(sino_id)
    astra.projector.delete(projector_id)
    astra.algorithm.delete(alg_id)
    return rec


# Task 1: Astra filtered backprojection (FBP)
def astra_fbp(sinogram, angles, output_shape):
    n = output_shape[0]
    det_count = sinogram.shape[1]
    vol_geom = astra.create_vol_geom(n, n)
    proj_geom = astra.create_proj_geom('parallel', 1.0, det_count, angles)
    projector_id = astra.create_projector('strip', proj_geom, vol_geom)
    rec_id = astra.data2d.create('-vol', vol_geom)
    sino_id = astra.data2d.create('-sino', proj_geom, sinogram.astype(np.float32))
    cfg = astra.astra_dict('FBP')
    cfg['ReconstructionDataId'] = rec_id
    cfg['ProjectionDataId'] = sino_id
    cfg['ProjectorId'] = projector_id
    alg_id = astra.algorithm.create(cfg)
    astra.algorithm.run(alg_id)
    rec = astra.data2d.get(rec_id)
    astra.data2d.delete(rec_id)
    astra.data2d.delete(sino_id)
    astra.projector.delete(projector_id)
    astra.algorithm.delete(alg_id)
    return rec


# Task 3: Matrix-free Krylov Tikhonov (zero-order / first-order)
def matrix_free_tikhonov_recon(sinogram, angles, output_shape, alpha=0.01, order=0, maxiter=80, rtol=1e-5, return_info=False):
    from scipy.sparse.linalg import LinearOperator, cg

    n = output_shape[0]
    n_angles = len(angles)
    det_count = sinogram.shape[1]
    vol_geom = astra.create_vol_geom(n, n)
    proj_geom = astra.create_proj_geom('parallel', 1.0, det_count, angles)
    projector_id = astra.create_projector('strip', proj_geom, vol_geom)

    def A(x):
        x_img = x.reshape((n, n)).astype(np.float32)
        sino_id, sino = astra.create_sino(x_img, projector_id)
        astra.data2d.delete(sino_id)
        return sino.ravel()

    def AT(y):
        y_sino = y.reshape((n_angles, det_count)).astype(np.float32)
        sino_id = astra.data2d.create('-sino', proj_geom, y_sino)
        rec_id = astra.data2d.create('-vol', vol_geom)
        cfg = astra.astra_dict('BP')
        cfg['ReconstructionDataId'] = rec_id
        cfg['ProjectionDataId'] = sino_id
        cfg['ProjectorId'] = projector_id
        alg_id = astra.algorithm.create(cfg)
        astra.algorithm.run(alg_id)
        rec = astra.data2d.get(rec_id)
        astra.data2d.delete(rec_id)
        astra.data2d.delete(sino_id)
        astra.algorithm.delete(alg_id)
        return rec.ravel()

    g = sinogram.ravel().astype(np.float32)

    def matvec(x):
        if order == 0:
            reg_term = alpha * x
        elif order == 1:
            x_img = x.reshape((n, n))
            reg_term = -alpha * laplace(x_img).ravel()
        else:
            raise ValueError('order must be 0 or 1')
        return AT(A(x)) + reg_term

    A_op = LinearOperator((n * n, n * n), matvec=matvec)
    rhs = AT(g)
    x, info = cg(A_op, rhs, maxiter=maxiter, rtol=rtol)

    astra.projector.delete(projector_id)

    if return_info:
        return x.reshape((n, n)), info
    return x.reshape((n, n))


# Task 4: Explicit Radon matrix for SVD
def radon_matrix(output_shape, angles):
    n = output_shape[0]
    det_count = int(np.ceil(np.sqrt(2) * n))
    vol_geom = astra.create_vol_geom(n, n)
    proj_geom = astra.create_proj_geom('parallel', 1.0, det_count, angles)
    projector_id = astra.create_projector('strip', proj_geom, vol_geom)
    A = []
    for i in range(n * n):
        e = np.zeros((n, n), dtype=np.float32)
        e.flat[i] = 1.0
        sino_id, sino = astra.create_sino(e, projector_id)
        astra.data2d.delete(sino_id)
        A.append(sino.ravel())
    A = np.stack(A, axis=1)
    astra.projector.delete(projector_id)
    return A


# Task 3 baseline: Frequency-domain Tikhonov
def tikhonov_recon(sinogram, theta, output_shape, alpha=0.01):
    sino_dct = dct(sinogram, axis=0, norm='ortho')
    filter_ = 1 / (1 + alpha * (np.arange(sinogram.shape[0], dtype=np.float32)[:, None] ** 2))
    sino_dct_reg = sino_dct * filter_
    sino_reg = idct(sino_dct_reg, axis=0, norm='ortho')
    return astra_fbp(sino_reg, theta, output_shape)


# Part B: Isotropic sinogram inpainting
def inpaint_sinogram(sino, mask, alpha=0.2, n_iter=200):
    g = sino.copy().astype(np.float32)
    for _ in range(n_iter):
        lap = laplace(g)
        g[~mask] += alpha * lap[~mask]
        g[mask] = sino[mask]
    return g


# Part B: TV sinogram inpainting
def inpaint_sinogram_tv(sino, mask, alpha=0.2, n_iter=200, lam=0.1):
    g = sino.copy().astype(np.float32)
    for _ in range(n_iter):
        gradx = np.zeros_like(g)
        grady = np.zeros_like(g)
        gradx[:-1, :] = g[1:, :] - g[:-1, :]
        grady[:, :-1] = g[:, 1:] - g[:, :-1]
        div = np.zeros_like(g)
        div[:-1, :] += gradx[:-1, :]
        div[1:, :] -= gradx[:-1, :]
        div[:, :-1] += grady[:, :-1]
        div[:, 1:] -= grady[:, :-1]
        g[~mask] += alpha * (div[~mask] - lam * g[~mask])
        g[mask] = sino[mask]
    return g


# Task 4: Haar denoising with controllable threshold range
def haar_denoise(img, threshold=0.1, levels_to_threshold=None):
    coeffs = pywt.wavedec2(img, 'haar', level=2)
    coeffs_thresh = [coeffs[0]]
    total_detail_levels = len(coeffs) - 1

    if levels_to_threshold is None:
        levels_to_threshold = list(range(1, total_detail_levels + 1))

    for detail_idx in range(1, len(coeffs)):
        level_number = total_detail_levels - detail_idx + 1
        if level_number in levels_to_threshold:
            coeffs_thresh.append(tuple(pywt.threshold(c, threshold, mode='soft') for c in coeffs[detail_idx]))
        else:
            coeffs_thresh.append(coeffs[detail_idx])

    return pywt.waverec2(coeffs_thresh, 'haar')


# Task 5: ISTA sparse reconstruction
def ista(A, AT, g, lam, n_iter=50, tol=1e-5):
    """ISTA with convergence criterion.
    
    Parameters:
    - n_iter: maximum iteration count
    - tol: convergence tolerance (relative change in x)
    
    Stopping criterion: halt when ||x^{k+1} - x^k|| / (||x^k|| + eps) < tol
    This ensures small relative changes in the iterate (good stopping criterion).
    """
    x = np.zeros(A.shape[1], dtype=np.float32)
    apply_AT = AT if callable(AT) else (lambda z: AT @ z)
    t = 1.0 / (np.linalg.norm(A, ord=2) ** 2 + 1e-8)
    
    for iter_idx in range(n_iter):
        x_old = x.copy()
        
        # ISTA update: gradient step + soft threshold + non-negativity
        x = pywt.threshold(x + t * apply_AT(g - A @ x), t * lam, mode='soft')
        x = np.maximum(x, 0)
        
        # Stopping criterion: check relative change in x
        rel_change = np.linalg.norm(x - x_old) / (np.linalg.norm(x_old) + 1e-8)
        if rel_change < tol:
            # Convergence achieved
            print(f"ISTA converged at iteration {iter_idx}")
            break
    
    return x


def image_metrics(ref, img):
    return {
        'mse': mean_squared_error(ref, img),
        'psnr': peak_signal_noise_ratio(ref, img),
        'ssim': structural_similarity(ref, img, data_range=ref.max() - ref.min())
    }

In [ ]:
# Task 1: Load phantom and core Radon/FBP pipeline
phantom_path = Path('SLphan.npy')
if not phantom_path.exists():
    raise FileNotFoundError(f'Cannot find phantom file: {phantom_path.resolve()}')

phantom = np.load(phantom_path)
show_img(phantom, 'Task1_Original_Phantom', save_tag='task1_original_phantom')

angles = np.linspace(0.0, np.pi, 180, endpoint=False).astype(np.float32)
sinogram = astra_radon(phantom, angles)
show_img(sinogram, 'Task1_Full_Sinogram', save_tag='task1_full_sinogram')
print(f'Task 1 Answer: sinogram size = {sinogram.shape} (angles x detector samples)')

recon_bp = astra_bp(sinogram, angles, phantom.shape)
recon_fbp = astra_fbp(sinogram, angles, phantom.shape)
show_img(recon_bp, 'Task1_Unfiltered_Backprojection', save_tag='task1_unfiltered_bp')
show_img(recon_fbp, 'Task1_FBP_Reconstruction', save_tag='task1_fbp_reconstruction')
print(f'Task 1 Answer: backprojected image size = {recon_bp.shape}')

# Task 1: Noise sensitivity of FBP
print('\n--- Task 1: FBP noise sensitivity ---')
noise_levels_task1 = [0.0, 0.02, 0.05, 0.1]
fbp_noise_curve = []
for sigma in noise_levels_task1:
    np.random.seed(0)
    sino_noisy = sinogram + np.random.normal(0, sigma, sinogram.shape).astype(np.float32)
    recon_noisy = astra_fbp(sino_noisy, angles, phantom.shape)
    mse_val = mean_squared_error(phantom, recon_noisy)
    psnr_val = peak_signal_noise_ratio(phantom, recon_noisy)
    fbp_noise_curve.append((sigma, mse_val, psnr_val))
    print(f'Noise sigma={sigma:.2f}: MSE={mse_val:.6f}, PSNR={psnr_val:.2f} dB')

plt.figure(figsize=(6, 4))
plt.plot([x[0] for x in fbp_noise_curve], [x[1] for x in fbp_noise_curve], 'o-', linewidth=2)
plt.xlabel('Noise sigma')
plt.ylabel('FBP MSE')
plt.title('Task 1: Noise vs Reconstruction Error')
plt.grid(True, alpha=0.3)
save_current_figure('task1_noise_error_curve')

# Keep one noisy FBP for Task 4
np.random.seed(0)
recon_fbp_noisy = recon_fbp + np.random.normal(0, 0.1, recon_fbp.shape).astype(np.float32)
show_img(recon_fbp_noisy, 'Task4_Noisy_FBP_Input', save_tag='task4_noisy_fbp_input')

# Task 2: Explicit Radon matrix and SVD spectrum study
print('\n--- Task 2: Explicit matrix and SVD study ---')
n_svd = 32
angles_45_full = np.linspace(0.0, np.pi, 45, endpoint=False).astype(np.float32)
angles_90_full = np.linspace(0.0, np.pi, 90, endpoint=False).astype(np.float32)
angles_45_limited = np.linspace(0.0, np.pi / 2, 45, endpoint=False).astype(np.float32)

A_45_full = radon_matrix((n_svd, n_svd), angles_45_full)
A_90_full = radon_matrix((n_svd, n_svd), angles_90_full)
A_45_limited = radon_matrix((n_svd, n_svd), angles_45_limited)

_, S_45_full, _ = svd(A_45_full, full_matrices=False)
_, S_90_full, _ = svd(A_90_full, full_matrices=False)
_, S_45_limited, _ = svd(A_45_limited, full_matrices=False)

plt.figure(figsize=(8, 5))
plt.semilogy(S_90_full / S_90_full[0], label='90 views, 0-180 deg', linewidth=2)
plt.semilogy(S_45_full / S_45_full[0], label='45 views, 0-180 deg', linewidth=2)
plt.semilogy(S_45_limited / S_45_limited[0], label='45 views, 0-90 deg', linewidth=2)
plt.xlabel('Singular value index')
plt.ylabel('Normalized singular value')
plt.title('Task 2: SVD spectra under different geometries')
plt.grid(True, alpha=0.3)
plt.legend()
save_current_figure('task2_svd_spectra')

print('Task 2 Answer (i): fewer projections (90 -> 45) shift the spectrum down but keep similar decay trend.')
print('Task 2 Answer (ii): limited-angle (0-90 deg) collapses singular values much faster (more severe ill-posedness).')

# Task 3: Matrix-free regularized LS (FBP vs zero/first-order) with noisy data
print('\n--- Task 3: Matrix-free regularized least-squares comparisons ---')

def best_tikhonov_order(sino_in, ang_in, shape, ref, order, alpha_grid):
    best = {'alpha': None, 'mse': np.inf, 'recon': None}
    for alpha in alpha_grid:
        rec = matrix_free_tikhonov_recon(sino_in, ang_in, shape, alpha=alpha, order=order)
        mse_val = mean_squared_error(ref, rec)
        if mse_val < best['mse']:
            best = {'alpha': alpha, 'mse': mse_val, 'recon': rec}
    return best

alpha_grid_0 = [5e-4, 1e-3, 5e-3, 1e-2]
alpha_grid_1 = [5e-4, 1e-3, 5e-3, 1e-2]
noise_sigma_task3 = 0.05

task3_results = {}

for case_name, case_angles in [
    ('few_angles_full_range', angles_45_full),
    ('limited_angles', angles_45_limited),
]:
    sino_case = astra_radon(phantom, case_angles)
    np.random.seed(0)
    sino_case_noisy = sino_case + np.random.normal(0, noise_sigma_task3, sino_case.shape).astype(np.float32)

    recon_fbp_case = astra_fbp(sino_case_noisy, case_angles, phantom.shape)
    best_o0 = best_tikhonov_order(sino_case_noisy, case_angles, phantom.shape, phantom, order=0, alpha_grid=alpha_grid_0)
    best_o1 = best_tikhonov_order(sino_case_noisy, case_angles, phantom.shape, phantom, order=1, alpha_grid=alpha_grid_1)

    task3_results[case_name] = {
        'fbp': recon_fbp_case,
        'o0': best_o0,
        'o1': best_o1,
        'angles': case_angles,
    }

    print(f"Case={case_name}")
    print(f"  FBP    : MSE={mean_squared_error(phantom, recon_fbp_case):.6f}, PSNR={peak_signal_noise_ratio(phantom, recon_fbp_case):.2f}")
    print(f"  O0-Tikh: alpha={best_o0['alpha']:.4g}, MSE={best_o0['mse']:.6f}, PSNR={peak_signal_noise_ratio(phantom, best_o0['recon']):.2f}")
    print(f"  O1-Tikh: alpha={best_o1['alpha']:.4g}, MSE={best_o1['mse']:.6f}, PSNR={peak_signal_noise_ratio(phantom, best_o1['recon']):.2f}")

    show_img(recon_fbp_case, f'Task3_{case_name}_FBP_noisy', save_tag=f'task3_{case_name}_fbp')
    show_img(best_o0['recon'], f'Task3_{case_name}_ZeroOrder_Tikh', save_tag=f'task3_{case_name}_o0')
    show_img(best_o1['recon'], f'Task3_{case_name}_FirstOrder_Tikh', save_tag=f'task3_{case_name}_o1')

# Keep a representative best full-angle Krylov result for summary metrics
best_alpha = best_tikhonov_order(sinogram, angles, phantom.shape, phantom, order=0, alpha_grid=alpha_grid_0)['alpha']
recon_krylov = matrix_free_tikhonov_recon(sinogram, angles, phantom.shape, alpha=best_alpha, order=0)
show_img(recon_krylov, f'Task3_FullAngle_Krylov_alpha_{best_alpha:.4g}', save_tag='task3_fullangle_krylov_best')

In [ ]:
# Task 5: ISTA sparse tomography with varying noise and geometries
print('\n--- Task 5: ISTA sparse reconstruction ---')
phantom64 = phantom[:64, :64]

angles_ista_full = np.linspace(0.0, np.pi, 45, endpoint=False).astype(np.float32)
angles_ista_limited = np.linspace(0.0, np.pi / 2, 45, endpoint=False).astype(np.float32)

A_ista_full = radon_matrix((64, 64), angles_ista_full)
A_ista_limited = radon_matrix((64, 64), angles_ista_limited)

ista_results = {}
for geom_name, A_ista, ang_ista in [
    ('few_angles_full_range', A_ista_full, angles_ista_full),
    ('limited_angles', A_ista_limited, angles_ista_limited),
]:
    sino64 = astra_radon(phantom64, ang_ista)
    ista_results[geom_name] = {}

    for noise_level in [0.0, 0.05, 0.1]:
        np.random.seed(0)
        sino64_noisy = sino64 + np.random.normal(0, noise_level, sino64.shape).astype(np.float32)
        g_noisy = sino64_noisy.ravel()

        x_ista = ista(A_ista, A_ista.T, g_noisy, lam=0.1, n_iter=50)
        recon_ista = x_ista.reshape(64, 64)

        mse_ista = mean_squared_error(phantom64, recon_ista)
        psnr_ista = peak_signal_noise_ratio(phantom64, recon_ista)
        ista_results[geom_name][noise_level] = {
            'recon': recon_ista,
            'mse': mse_ista,
            'psnr': psnr_ista,
        }

        print(
            f"Geometry={geom_name}, Noise={noise_level:.2f}: "
            f"MSE={mse_ista:.6f}, PSNR={psnr_ista:.2f} dB"
        )

show_img(
    ista_results['few_angles_full_range'][0.1]['recon'],
    'Task5_ISTA_FewAngles_Noise0.1',
    save_tag='task5_ista_fewangles_noise01'
)
show_img(
    ista_results['limited_angles'][0.1]['recon'],
    'Task5_ISTA_LimitedAngles_Noise0.1',
    save_tag='task5_ista_limitedangles_noise01'
)

# Keep reference variable names for later summary
recon_ista = ista_results['few_angles_full_range'][0.0]['recon']
recon_ista_noisy = ista_results['few_angles_full_range'][0.1]['recon']

In [ ]:
# Task 4: Haar wavelet denoising and threshold-range study
print(chr(10) + '--- Task 4: Haar wavelet denoising ---')

coeffs = pywt.wavedec2(recon_fbp_noisy, 'haar', level=2)
recon_exact = pywt.waverec2(coeffs, 'haar')
recon_exact = recon_exact[:recon_fbp_noisy.shape[0], :recon_fbp_noisy.shape[1]]
print(f'Task 4 check: inverse wavelet reconstruction MSE={mean_squared_error(recon_fbp_noisy, recon_exact):.10f}')

for thresh in [0.05, 0.1, 0.2]:
    denoised_all = haar_denoise(recon_fbp_noisy, threshold=thresh, levels_to_threshold=[1, 2])
    denoised_l1 = haar_denoise(recon_fbp_noisy, threshold=thresh, levels_to_threshold=[1])
    show_img(denoised_all, f'Task4_Denoised_all_levels_thresh_{thresh}', save_tag=f'task4_denoised_all_t{thresh}')
    show_img(denoised_l1, f'Task4_Denoised_level1_only_thresh_{thresh}', save_tag=f'task4_denoised_l1_t{thresh}')

# Task 4: coefficient visualization
coeffs_vis = pywt.wavedec2(recon_fbp_noisy, 'haar', level=2)
detail_levels = len(coeffs_vis) - 1
plt.figure(figsize=(4 * (detail_levels + 1), 3))
plt.subplot(1, detail_levels + 1, 1)
plt.imshow(coeffs_vis[0], cmap='gray')
plt.title('Approximation')
for i in range(detail_levels):
    plt.subplot(1, detail_levels + 1, i + 2)
    plt.imshow(coeffs_vis[i + 1][0], cmap='gray')
    plt.title(f'Detail L{detail_levels - i}')
save_current_figure('task4_wavelet_coefficients')

# Task 4 supplement: compare decomposition depths 2, 4, 7 at a fixed threshold
print(chr(10) + '--- Task 4 supplement: level comparison (2, 4, 7) ---')

def haar_denoise_depth(img, threshold=0.1, wavelet_level=2):
    coeffs_depth = pywt.wavedec2(img, 'haar', level=wavelet_level)
    coeffs_thresh = [coeffs_depth[0]]
    for detail in coeffs_depth[1:]:
        coeffs_thresh.append(tuple(pywt.threshold(c, threshold, mode='soft') for c in detail))
    recon_depth = pywt.waverec2(coeffs_thresh, 'haar')
    return recon_depth[:img.shape[0], :img.shape[1]]

level_compare_threshold = 0.1
level_compare_results = {}
plt.figure(figsize=(12, 3.8))
for panel_idx, wavelet_level in enumerate([2, 4, 7], start=1):
    recon_level = haar_denoise_depth(recon_fbp_noisy, threshold=level_compare_threshold, wavelet_level=wavelet_level)
    level_compare_results[wavelet_level] = {
        'recon': recon_level,
        'mse': mean_squared_error(phantom, recon_level),
        'psnr': peak_signal_noise_ratio(phantom, recon_level),
    }
    plt.subplot(1, 3, panel_idx)
    plt.imshow(recon_level, cmap='gray')
    plt.title(f'Level={wavelet_level}')
    plt.axis('off')
    print(
        f'  Level={wavelet_level}, threshold={level_compare_threshold:.2f}: ' +
        f"MSE={level_compare_results[wavelet_level]['mse']:.6f}, " +
        f"PSNR={level_compare_results[wavelet_level]['psnr']:.2f} dB"
    )

plt.suptitle('Task 4 supplement: fixed-threshold Haar denoising at depths 2, 4, 7')
plt.tight_layout()
level_compare_path = EXPORT_DIR / 'task4_level_comparison.png'
plt.savefig(level_compare_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved figure: {level_compare_path}')
print('Observation: deeper decompositions suppress high-frequency noise more strongly, but the level-7 result is visibly smoother and more prone to over-smoothing than level-2/4.')

# Task 3 baseline: frequency-domain Tikhonov
recon_tikh = tikhonov_recon(sinogram, angles, phantom.shape, alpha=0.01)
show_img(recon_tikh, 'Task3_FrequencyDomain_Tikhonov', save_tag='task3_frequency_domain_tikhonov')

# Part B (Advanced Topic 1): sinogram inpainting
print(chr(10) + '--- Part B: Sinogram inpainting (undersampled and limited-angle) ---')

# --- Undersampled case ---
mask_undersampled = np.ones_like(sinogram, dtype=bool)
mask_undersampled[:, ::2] = False

# BEFORE inpainting: corrupted sinogram (zeros at missing detectors) and naive FBP
sino_corrupted_under = sinogram.copy()
sino_corrupted_under[~mask_undersampled] = 0.0
recon_corrupted_under = astra_fbp(sino_corrupted_under, angles, phantom.shape)
show_img(sino_corrupted_under, 'PartB_Corrupted_Sinogram_Undersampled_Before', save_tag='partb_corrupted_sino_undersampled')
show_img(recon_corrupted_under, 'PartB_Naive_FBP_Undersampled_Before', save_tag='partb_naive_fbp_undersampled')

# AFTER inpainting
sino_inpainted = inpaint_sinogram(sinogram, mask_undersampled, alpha=0.2, n_iter=200)
recon_inpainted = astra_fbp(sino_inpainted, angles, phantom.shape)
show_img(sino_inpainted, 'PartB_Inpainted_Sinogram_Undersampled', save_tag='partb_inpainted_sino_undersampled')
show_img(recon_inpainted, 'PartB_Reconstruction_Undersampled', save_tag='partb_recon_undersampled')

# --- Limited-angle case ---
mask_limited = np.ones_like(sinogram, dtype=bool)
mask_limited[90:, :] = False

# BEFORE inpainting: corrupted limited-angle sinogram and naive FBP
sino_corrupted_limited = sinogram.copy()
sino_corrupted_limited[~mask_limited] = 0.0
recon_corrupted_limited = astra_fbp(sino_corrupted_limited, angles, phantom.shape)
show_img(sino_corrupted_limited, 'PartB_Corrupted_Sinogram_LimitedAngle_Before', save_tag='partb_corrupted_sino_limited')
show_img(recon_corrupted_limited, 'PartB_Naive_FBP_LimitedAngle_Before', save_tag='partb_naive_fbp_limited')

# AFTER TV inpainting
sino_inpainted_limited = inpaint_sinogram_tv(sinogram, mask_limited, alpha=0.2, n_iter=200, lam=0.1)
recon_inpainted_limited = astra_fbp(sino_inpainted_limited, angles, phantom.shape)
show_img(sino_inpainted_limited, 'PartB_TV_Inpainted_Sinogram_LimitedAngle', save_tag='partb_tv_inpainted_sino_limited')
show_img(recon_inpainted_limited, 'PartB_Reconstruction_LimitedAngle', save_tag='partb_recon_limited')

# Part B: image-domain denoising after inpainting reconstruction
recon_inpainted_denoised = haar_denoise(recon_inpainted, threshold=0.05, levels_to_threshold=[1, 2])
show_img(recon_inpainted_denoised, 'PartB_Reconstruction_Undersampled_Denoised', save_tag='partb_recon_undersampled_denoised')

print('Part B comparison (undersampled):')
print(f'  Naive FBP (no inpaint)  MSE={mean_squared_error(phantom, recon_corrupted_under):.6f}, PSNR={peak_signal_noise_ratio(phantom, recon_corrupted_under):.2f}')
print(f'  Inpainted recon         MSE={mean_squared_error(phantom, recon_inpainted):.6f}, PSNR={peak_signal_noise_ratio(phantom, recon_inpainted):.2f}')
print(f'  +Wavelet denoise        MSE={mean_squared_error(phantom, recon_inpainted_denoised):.6f}, PSNR={peak_signal_noise_ratio(phantom, recon_inpainted_denoised):.2f}')


In [ ]:
# Final quantitative summary for report writing
def _match_shape(ref, img):
    r, c = ref.shape
    r2, c2 = img.shape
    dr, dc = r - r2, c - c2
    if dr < 0:
        img = img[(-dr) // 2:r2 + (dr - dr // 2), :]
    elif dr > 0:
        img = np.pad(img, ((dr // 2, dr - dr // 2), (0, 0)), mode='constant')
    if dc < 0:
        img = img[:, (-dc) // 2:c2 + (dc - dc // 2)]
    elif dc > 0:
        img = np.pad(img, ((0, 0), (dc // 2, dc - dc // 2)), mode='constant')
    return img

recon_bp_ = _match_shape(phantom, recon_bp)
recon_tikh_ = _match_shape(phantom, recon_tikh)
recon_krylov_ = _match_shape(phantom, recon_krylov)
recon_inpainted_ = _match_shape(phantom, recon_inpainted)
recon_inpainted_denoised_ = _match_shape(phantom, recon_inpainted_denoised)

mse_bp = mean_squared_error(phantom, recon_bp_)
psnr_bp = peak_signal_noise_ratio(phantom, recon_bp_)
ssim_bp = structural_similarity(phantom, recon_bp_, data_range=phantom.max() - phantom.min())

mse_fbp = mean_squared_error(phantom, recon_fbp)
psnr_fbp = peak_signal_noise_ratio(phantom, recon_fbp)
ssim_fbp = structural_similarity(phantom, recon_fbp, data_range=phantom.max() - phantom.min())

mse_krylov = mean_squared_error(phantom, recon_krylov_)
psnr_krylov = peak_signal_noise_ratio(phantom, recon_krylov_)
ssim_krylov = structural_similarity(phantom, recon_krylov_, data_range=phantom.max() - phantom.min())

mse_tikh = mean_squared_error(phantom, recon_tikh_)
psnr_tikh = peak_signal_noise_ratio(phantom, recon_tikh_)
ssim_tikh = structural_similarity(phantom, recon_tikh_, data_range=phantom.max() - phantom.min())

mse_inpaint = mean_squared_error(phantom, recon_inpainted_)
psnr_inpaint = peak_signal_noise_ratio(phantom, recon_inpainted_)
ssim_inpaint = structural_similarity(phantom, recon_inpainted_, data_range=phantom.max() - phantom.min())

mse_inpaint_denoised = mean_squared_error(phantom, recon_inpainted_denoised_)
psnr_inpaint_denoised = peak_signal_noise_ratio(phantom, recon_inpainted_denoised_)
ssim_inpaint_denoised = structural_similarity(phantom, recon_inpainted_denoised_, data_range=phantom.max() - phantom.min())

print(f'Best full-angle zero-order Krylov alpha: {best_alpha:.4g}')
print('BP:                 MSE=%.6f, PSNR=%.2f, SSIM=%.4f' % (mse_bp, psnr_bp, ssim_bp))
print('FBP:                MSE=%.6f, PSNR=%.2f, SSIM=%.4f' % (mse_fbp, psnr_fbp, ssim_fbp))
print('Krylov (order=0):   MSE=%.6f, PSNR=%.2f, SSIM=%.4f' % (mse_krylov, psnr_krylov, ssim_krylov))
print('Tikhonov (DCT):     MSE=%.6f, PSNR=%.2f, SSIM=%.4f' % (mse_tikh, psnr_tikh, ssim_tikh))
print('Inpainted:          MSE=%.6f, PSNR=%.2f, SSIM=%.4f' % (mse_inpaint, psnr_inpaint, ssim_inpaint))
print('Inpaint+denoised:   MSE=%.6f, PSNR=%.2f, SSIM=%.4f' % (mse_inpaint_denoised, psnr_inpaint_denoised, ssim_inpaint_denoised))

print('\nTask 3 case-wise comparison (noisy, few angles vs limited angles):')
for case_name, case_data in task3_results.items():
    fbp_case = case_data['fbp']
    o0_case = case_data['o0']['recon']
    o1_case = case_data['o1']['recon']
    print(f'  {case_name}:')
    print('    FBP      MSE=%.6f, PSNR=%.2f' % (mean_squared_error(phantom, fbp_case), peak_signal_noise_ratio(phantom, fbp_case)))
    print('    O0-Tikh  MSE=%.6f, PSNR=%.2f (alpha=%s)' % (
        mean_squared_error(phantom, o0_case),
        peak_signal_noise_ratio(phantom, o0_case),
        format(case_data['o0']['alpha'], '.4g')
    ))
    print('    O1-Tikh  MSE=%.6f, PSNR=%.2f (alpha=%s)' % (
        mean_squared_error(phantom, o1_case),
        peak_signal_noise_ratio(phantom, o1_case),
        format(case_data['o1']['alpha'], '.4g')
    ))

print('\nExported figure files are saved automatically to:', EXPORT_DIR.resolve())

In [ ]:
# Flush and close the output log, restore stdout
sys.stdout.flush()
_tee._file.flush()
_tee._file.close()
sys.stdout = _tee._screen
print("All text output saved to: notebook_output_log.txt")
